# 05 · Embeddings e indexação dos precedentes

Este notebook gera os vetores dos trechos do notebook 04 e grava a base no PostgreSQL/pgvector, no schema próprio `lex_precedentes`. Ele segue o mesmo padrão das bases de legislação e de jurisprudência.

| Item | Decisão |
|---|---|
| Modelo | **BGE-M3**, na mesma revisão usada nas outras bases (`src/integracao/recuperacao.py`), para que pergunta e trecho fiquem no mesmo espaço vetorial |
| Vetores | 1024 dimensões, normalizados; a semelhança é o cosseno |
| Versão | Identificada pelo **hash do manifesto**: modelo, revisão, hash dos trechos e hash dos vetores na ordem. Qualquer mudança gera outra versão |
| Importação | Numa transação. Depois, os textos e os vetores são relidos do banco e os hashes conferidos. Só então a versão fica `validado`, e a anterior passa a `substituido` |
| Tabelas | `datasets` (manifesto e estado); `embeddings` (um trecho por linha); `registros` (o registro inteiro, incluindo os que estão fora da busca, para a entrega e para dizer que um enunciado foi cancelado) |
| Índice | Busca **exata**, sem índice aproximado: com cerca de 1,3 mil vetores, a varredura completa é rápida e não perde resultados. Mais um índice para a busca por palavras (`lex_precedentes.lexico`), com o texto normalizado e **sem stopwords** (seção 3) |

### Dados desta etapa

| | Caminho no repositório | Formato | Quantidade | Versionado? |
|---|---|---|---|---|
| Entrada | `data/processed/precedentes/chunks_precedentes.jsonl` | JSONL (um registro JSON por linha) | 1.268 trechos | fora do Git |
| Entrada | `data/processed/precedentes/precedentes_tst.jsonl` | JSONL (um registro JSON por linha) | 1.493 registros | fora do Git |
| Saída (cache) | `data/processed/precedentes/vetores_bge_m3.npy` | NumPy (.npy), float32 | matriz 1.268 × 1.024 | fora do Git |
| Saída (banco) | `PostgreSQL no Docker, schema lex_precedentes` | tabelas | datasets (1 versão validada), embeddings (1.268 linhas), registros (1.493), lexico (1.268) | fora do Git (banco) |
| Relatórios | `reports/precedentes/indexacao/manifesto_indexacao.json` | JSON | manifesto da versão (hashes e contagens) | sim |

**Formato:** a base estruturada é gravada em **JSONL** (texto, um registro JSON por linha), e não em CSV, porque cada registro tem campos com listas (itens da súmula, páginas, assuntos, histórico). Os arquivos **CSV** do projeto são os relatórios em `reports/precedentes/`. Os arquivos de dados ficam em `data/`, dentro da pasta do repositório, mas **fora do Git** (`.gitignore`): contêm texto integral e são recriados pelos notebooks.

In [1]:
from pathlib import Path
import json, os, sys, hashlib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/precedentes').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault('HF_HUB_OFFLINE', '1')
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 250); pd.set_option('display.max_columns', None)
PROC = ROOT / 'data/processed/precedentes'
REGISTROS = PROC / 'precedentes_tst.jsonl'
TRECHOS = PROC / 'chunks_precedentes.jsonl'
REL = ROOT / 'reports/precedentes'

import numpy as np, psycopg
from sentence_transformers import SentenceTransformer
from src.integracao.recuperacao import MODELO, REVISAO
from src.precedentes import indexacao as ix
from src.storage.pgvector_store import load_database_url
trechos = [json.loads(l) for l in TRECHOS.read_text(encoding='utf-8').splitlines() if l.strip()]
registros = [json.loads(l) for l in REGISTROS.read_text(encoding='utf-8').splitlines() if l.strip()]
man_ch = json.loads((REL / 'chunking/manifesto_chunking.json').read_text(encoding='utf-8'))
sha_tr = hashlib.sha256(TRECHOS.read_bytes()).hexdigest()
sha_reg = hashlib.sha256(REGISTROS.read_bytes()).hexdigest()
assert sha_tr == man_ch['sha256'] and sha_reg == man_ch['registros_jsonl_sha256'], 'trechos ou registros diferentes do manifesto do 04'
(REL / 'indexacao').mkdir(parents=True, exist_ok=True)
print(f'{len(trechos)} trechos · {len(registros)} registros · conferidos com o manifesto do 04 · modelo {MODELO}@{REVISAO[:8]}')

1268 trechos · 1493 registros · conferidos com o manifesto do 04 · modelo BAAI/bge-m3@5617a9f6


## 1. Vetores

Os vetores são calculados no computador local, na CPU, porque são poucos trechos. Eles ficam num cache em `data/processed/precedentes/`. Uma nova execução só recalcula se os textos mudarem.

> **Lê:** `data/processed/precedentes/chunks_precedentes.jsonl` (1.268 trechos)  
> **Grava:** `data/processed/precedentes/vetores_bge_m3.npy` (NumPy, 1.268 × 1.024, fora do Git)

In [2]:
import time
inicio = time.monotonic()
encoder = SentenceTransformer(MODELO, revision=REVISAO, device='cpu', local_files_only=True)
encoder.max_seq_length = 8192
vetores, do_cache = ix.vetorizar(trechos, encoder, cache=PROC / 'vetores_bge_m3.npy')
normas = np.linalg.norm(vetores, axis=1)
print(f'{vetores.shape[0]} vetores de {vetores.shape[1]} dimensões · {"do cache" if do_cache else f"calculados em {time.monotonic() - inicio:.0f} s"}')
print(f'norma: mín {normas.min():.4f} · máx {normas.max():.4f} (normalizados) · valores finitos: {bool(np.isfinite(vetores).all())}')

1268 vetores de 1024 dimensões · do cache
norma: mín 1.0000 · máx 1.0000 (normalizados) · valores finitos: True


## 2. Manifesto, importação e conferência

> **Lê:** os trechos, os vetores e os 1.493 registros  
> **Grava:** tabelas `lex_precedentes.datasets`, `lex_precedentes.embeddings` (1.268 linhas) e `lex_precedentes.registros` (1.493 linhas) no PostgreSQL; `reports/precedentes/indexacao/manifesto_indexacao.json`

In [3]:
manifesto = ix.manifesto(trechos, vetores, registros, MODELO, REVISAO, sha_tr, sha_reg, man_ch['limite_tokens'])
conexao = psycopg.connect(load_database_url(ROOT / '.env.pgvector.local'), autocommit=True)
dataset_id, ja_existia = ix.importar(conexao, trechos, vetores, registros, manifesto)
erros = ix.verificar(conexao, manifesto)
estados = conexao.execute('SELECT left(dataset_id, 12), estado, chunks_esperados, registros_esperados, atualizado_em '
                          'FROM lex_precedentes.datasets ORDER BY atualizado_em').fetchall()
print(f'Versão {dataset_id[:16]}… {"já estava validada" if ja_existia else "importada e validada"} · conferência: {erros or "ok"}')
display(pd.DataFrame(estados, columns=['dataset', 'estado', 'trechos', 'registros', 'atualizado_em']))
(REL / 'indexacao/manifesto_indexacao.json').write_text(json.dumps(manifesto, indent=2, ensure_ascii=False), encoding='utf-8')

Versão bdf048e69083b43d… já estava validada · conferência: ok


,dataset,estado,trechos,registros,atualizado_em
0,bdf048e69083,validado,1268,1493,2026-11-22 22:23:34.457941+00:00


## 3. Busca por palavras: normalização do texto, sem stopwords

A busca por palavras do PostgreSQL compara as palavras da pergunta com as do trecho. O texto jurídico tem várias formas de escrever a mesma coisa, e o índice padrão as trata como palavras diferentes.

**A normalização vale só para o índice e para a pergunta.** O texto guardado e entregue ao modelo continua o oficial, com "art. 7º", "Lei nº 8.036" e "§ 1º", porque o modelo e o validador citam a norma como ela está escrita.

| Regra | Exemplo | Problema que resolve |
|---|---|---|
| "§" e "§§" viram "paragrafo" e "paragrafos" | "§ 1º" → "paragrafo 1" | O índice padrão descartava o "§" |
| "art." e "arts." viram "artigo" e "artigos" | "art. 7º" → "artigo 7" | "art" e "artigo" eram palavras diferentes |
| "nº", "n.º", "n°" e "no." antes de número são removidos | "Lei nº 8.036" → "lei 8036" | — |
| O ordinal colado ao número é removido | "7º", "7°", "7o" e "1ª" → "7" e "1" | "7º" e "7" eram palavras diferentes |
| O ponto de milhar é removido, e a barra entre números é separada | "8.036/1990" → "8036 / 1990"; "Tema 1.046" → "1046" | — |
| O numeral romano isolado vira uma marca própria | "XIII" → "inc13" | O radicalizador do português transformava "xiii" em "xii", juntando os incisos XII e XIII |
| Tudo fica em minúsculas e sem acento | "não" = "nao" | — |

**Sem stopwords.** O índice padrão em português descarta palavras como "não", "de", "a" e "o". No texto jurídico, "não" muda o sentido da regra: "é devido" e "não é devido" ficavam iguais no índice. Por isso usamos uma configuração própria (`lex_precedentes.pt_sem_stopwords`), que mantém todas as palavras e só aplica o radicalizador, que reduz "trabalhadores" e "trabalhador" à mesma raiz.

**As palavras comuns não atrapalham porque cada palavra pesa pelo IDF**, como no BM25. "De" aparece em quase todos os trechos e pesa perto de zero; "338" ou "insalubridade" são raras e pesam muito. Nenhuma palavra é descartada, mas as frequentes quase não contam.

> **Lê:** `lex_precedentes.embeddings`  
> **Grava:** tabela `lex_precedentes.lexico` (1.268 linhas, texto normalizado) no PostgreSQL

In [4]:
from src.precedentes import normalizacao as nz
print('Trechos no índice de palavras:', ix.preparar_lexico(conexao, dataset_id), f'(normalização {nz.VERSAO})')
casos = ['art. 7º, XIII, da CF', 'Lei nº 8.036/1990', 'Tema 1.046 do STF', '§§ 2º e 3º', 'Não é devido o adicional']
comparacao = []
for t in casos:
    padrao = conexao.execute("SELECT to_tsvector('portuguese', %s)::text", (t,)).fetchone()[0]
    nosso = conexao.execute('SELECT to_tsvector(%s::regconfig, %s)::text', (nz.CONFIG, nz.normalizar(t))).fetchone()[0]
    comparacao.append({'texto': t, 'índice padrão (português)': padrao, 'texto normalizado': nz.normalizar(t), 'índice LexIA (sem stopwords)': nosso})
display(pd.DataFrame(comparacao))

Trechos no índice de palavras: 1268 (normalização normalizacao-lexica-v1)


,texto,índice padrão (português),texto normalizado,índice LexIA (sem stopwords)
0,"art. 7º, XIII, da CF",'7º':2 'art':1 'cf':5 'xii':3,"artigo 7, inc13, da cf",'7':2 'artig':1 'cf':5 'da':4 'inc13':3
1,Lei nº 8.036/1990,'/1990':4 '8.036':3 'lei':1 'nº':2,lei 8036 / 1990,'1990':3 '8036':2 'lei':1
2,Tema 1.046 do STF,'1.046':2 'stf':4 'tem':1,tema 1046 do stf,'1046':2 'do':3 'stf':4 'tem':1
3,§§ 2º e 3º,'2º':1 '3º':3,paragrafos 2 e 3,'2':2 '3':4 'e':3 'paragraf':1
4,Não é devido o adicional,'adicional':5 'dev':3 'é':2,nao e devido o adicional,'adicional':5 'dev':3 'e':2 'nao':1 'o':4


## 4. Testes de funcionamento

Estes testes não avaliam a qualidade da busca. Essa avaliação fica no notebook 06, com as perguntas da curadoria e o gabarito. Aqui só conferimos que o índice responde de forma coerente:
1. **Cada trecho acha a si mesmo.** O vetor de cada trecho, usado como consulta, tem que voltar em 1º lugar. Isso confere a gravação e a ordem dos vetores.
2. **Três perguntas de exemplo,** escritas para este teste (não são perguntas da curadoria), sobre temas em que já sabemos qual precedente decide. Mostramos em que posição o precedente esperado aparece na busca semântica.
3. **As mesmas perguntas na busca por palavras,** mais uma que cita a lei sem o ponto e sem o "nº" ("Lei 8036"). Ela tem que achar o texto que diz "Lei nº 8.036".

In [5]:
from pgvector.psycopg import register_vector
register_vector(conexao)
amostra = np.random.default_rng(42).choice(len(trechos), size=50, replace=False)
acertos = sum(ix.busca_semantica(conexao, dataset_id, vetores[i], k=1)[0][1] == trechos[i]['chunk_id'] for i in amostra)
print(f'1. Trecho acha a si mesmo: {acertos} de {len(amostra)}')

exemplos = [
    ('A empresa não apresentou os cartões de ponto no processo. Quem precisa provar as horas extras?', 'TST-SUM-338'),
    ('Trabalho à noite e a jornada passa das 5 da manhã. O adicional noturno vale para as horas depois das 5h?', 'TST-SUM-60'),
    ('A empresa olha dentro das bolsas dos funcionários todo dia, sem encostar neles. Isso gera dano moral?', 'TST-IRR-58'),
]
linhas = []
for pergunta, esperado in exemplos:
    q = encoder.encode([pergunta], normalize_embeddings=True)[0].astype(np.float32)
    achados = ix.busca_semantica(conexao, dataset_id, q, k=50)
    ordem_reg = list(dict.fromkeys(r for r, _, _ in achados))
    linhas.append(dict(esperado=esperado, posicao_entre_registros=(ordem_reg.index(esperado) + 1) if esperado in ordem_reg else '>50 trechos',
                       primeiro=ordem_reg[0], semelhanca_1o=round(achados[0][2], 3)))
display(pd.DataFrame(linhas))

print()
print('3. Busca por palavras (texto normalizado, sem stopwords, peso IDF):')
lex = []
for pergunta, esperado in exemplos + [('Lei 8036 saque do FGTS', 'TST-OJ-SDI1-42')]:
    achados = ix.busca_lexical(conexao, dataset_id, pergunta, k=20)
    ordem_reg = list(dict.fromkeys(r for r, _, _ in achados))
    lex.append(dict(pergunta=pergunta[:60], esperado=esperado,
                    posicao=(ordem_reg.index(esperado) + 1) if esperado in ordem_reg else '>20 trechos', primeiro=ordem_reg[0] if ordem_reg else None))
display(pd.DataFrame(lex))

1. Trecho acha a si mesmo: 50 de 50

3. Busca por palavras (texto normalizado, sem stopwords, peso IDF):


,esperado,posicao_entre_registros,primeiro,semelhanca_1o
0,TST-SUM-338,1,TST-SUM-338,0.621
1,TST-SUM-60,3,TST-OJ-SDI1-388,0.764
2,TST-IRR-58,1,TST-IRR-58,0.571


,pergunta,esperado,posicao,primeiro
0,A empresa não apresentou os cartões de ponto no processo. Qu,TST-SUM-338,1,TST-SUM-338
1,Trabalho à noite e a jornada passa das 5 da manhã. O adicion,TST-SUM-60,4,TST-IRR-92
2,"A empresa olha dentro das bolsas dos funcionários todo dia,",TST-IRR-58,1,TST-IRR-58
3,Lei 8036 saque do FGTS,TST-OJ-SDI1-42,1,TST-OJ-SDI1-42


**Leitura.**

No notebook 00, essas mesmas súmulas ficavam entre a 8ª e a 3.370ª posição quando disputavam com 20 mil acórdãos. Aqui, na base separada, elas disputam só com outros precedentes. O teste é só de funcionamento. A medida que vale (Hit@1 e Hit@3 com o gabarito, e a repetição do teste de posição com o texto oficial) fica no notebook 06.

## Conclusões

- **A base `lex_precedentes` está gravada e validada:**
  - trechos e vetores BGE-M3 na mesma revisão das outras bases;
  - registros inteiros para a entrega;
  - índice para a busca por palavras, com texto normalizado ("art. 7º" = "art. 7", "Lei nº 8.036" = "Lei 8036", incisos romanos separados), sem stopwords e com peso IDF.
- **A versão é reproduzível.** O hash do manifesto amarra modelo, revisão, trechos e vetores, e o JSONL de trechos é conferido com o manifesto do 04.
- **Próximos passos:**
  1. Notebook 06: avaliação da recuperação.
  2. Integração da base como terceira fonte da busca, com cota própria, e citação pelo nome no prompt.